# Polysemantic Feature Analysis

Tools for identifying and analyzing neurons/features that respond to
multiple unrelated concepts (polysemanticity), a key challenge in
mechanistic interpretability.

This notebook covers the `irtk.polysemantic_features` module.

## Why This Matters

Polysemanticity analysis quantifies how much each neuron responds to multiple unrelated concepts. Context clustering, activation decomposition, and interference matrices reveal the degree of feature overlap — directly measuring the problem that sparse autoencoders aim to solve.

**Key references:**
- [Bricken et al. (2023) "Towards Monosemanticity"](https://transformer-circuits.pub/2023/monosemantic-features/index.html) — Sparse autoencoders find interpretable features

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import polysemantic_features, sae

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Monosemanticity Ranking

Rank all dimensions by how monosemantic vs polysemantic they are.

In [ ]:
prompts = [
    "The Eiffel Tower is located in",
    "The capital of France is",
    "London is the capital of",
    "Berlin is a city in",
    "The president of the United States is",
    "Python is a programming language that",
    "The cat sat on the mat and",
    "In the year 2024 the world",
]
token_seqs = [model.to_tokens(p) for p in prompts]

result = polysemantic_features.monosemanticity_ranking(
    model, "blocks.6.hook_resid_post", token_seqs, top_k=10
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(result['all_scores'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(result['mean_score'], color='red', linestyle='--', label=f'Mean: {result["mean_score"]:.4f}')
ax.set_xlabel('Polysemanticity Score')
ax.set_ylabel('Count')
ax.set_title('Distribution of Polysemanticity Scores (Layer 6 Resid)')
ax.legend()
plt.tight_layout()
plt.show()

print("Most monosemantic dimensions:")
for dim, score in result['most_monosemantic'][:5]:
    print(f"  dim {dim}: score = {score:.4f}")

print("\nMost polysemantic dimensions:")
for dim, score in result['most_polysemantic'][:5]:
    print(f"  dim {dim}: score = {score:.4f}")

## 2. Polysemanticity Score

Detailed analysis of a single dimension's polysemanticity.

In [ ]:
# Analyze the most polysemantic dimension
poly_dim = result['most_polysemantic'][0][0]
mono_dim = result['most_monosemantic'][0][0]

poly_result = polysemantic_features.polysemanticity_score(
    model, "blocks.6.hook_resid_post", token_seqs, poly_dim
)
mono_result = polysemantic_features.polysemanticity_score(
    model, "blocks.6.hook_resid_post", token_seqs, mono_dim
)

print(f"Most polysemantic (dim {poly_dim}):")
for k, v in poly_result.items():
    print(f"  {k}: {v:.4f}")

print(f"\nMost monosemantic (dim {mono_dim}):")
for k, v in mono_result.items():
    print(f"  {k}: {v:.4f}")

## 3. Feature Context Clusters

Cluster the contexts where a feature activates to reveal polysemanticity.

In [ ]:
result = polysemantic_features.feature_context_clusters(
    model, "blocks.6.hook_resid_post", token_seqs, poly_dim,
    n_clusters=3, top_k=30
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if len(result['activation_values']) > 0:
    colors = ['steelblue', 'coral', 'seagreen']
    for i in range(len(result['activation_values'])):
        c = result['cluster_assignments'][i] if i < len(result['cluster_assignments']) else 0
        axes[0].scatter(i, result['activation_values'][i], color=colors[c % 3], s=30)
    axes[0].set_xlabel('Example Index')
    axes[0].set_ylabel('Activation Value')
    axes[0].set_title(f'Top Activations by Cluster (dim {poly_dim})')

axes[1].bar(range(len(result['cluster_sizes'])), result['cluster_sizes'], color=colors[:len(result['cluster_sizes'])])
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Size')
axes[1].set_title(f'Cluster Sizes (n_effective={result["n_effective_clusters"]})')

plt.tight_layout()
plt.show()

## 4. Feature Interference Matrix (SAE)

Measure decoder direction overlap between SAE features.

In [ ]:
# Train a small SAE
trained = sae.train_sae(
    model, "blocks.6.hook_resid_post", token_seqs,
    n_features=64, n_epochs=5, lr=1e-3, l1_coeff=1e-3
)
trained_sae = trained.sae

# Check interference among first 10 features
result = polysemantic_features.feature_interference_matrix(
    trained_sae, list(range(10))
)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(result['interference_matrix'], cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xlabel('Feature')
ax.set_ylabel('Feature')
ax.set_title(f'Feature Interference (orthogonality: {result["orthogonality_score"]:.3f})')
plt.colorbar(im, ax=ax, label='Cosine Similarity')
plt.tight_layout()
plt.show()

print(f"Max interference: {result['max_interference']:.4f}")
print(f"Mean interference: {result['mean_interference']:.4f}")

## 5. Activation Decomposition

Decompose an activation into its SAE features to reveal superposition.

In [ ]:
tokens = model.to_tokens("The Eiffel Tower is located in")
result = polysemantic_features.activation_decomposition(
    trained_sae, model, tokens, "blocks.6.hook_resid_post", pos=-1
)

print(f"Active features: {result['n_active']}")
print(f"Reconstruction error: {result['reconstruction_error']:.4f}")
print(f"Top feature fraction: {result['top_feature_fraction']:.2%}")

if result['active_features']:
    top_n = min(15, len(result['active_features']))
    feat_ids = [f[0] for f in result['active_features'][:top_n]]
    feat_vals = [f[1] for f in result['active_features'][:top_n]]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(range(top_n), feat_vals, color='steelblue')
    ax.set_yticks(range(top_n))
    ax.set_yticklabels([f'Feature {i}' for i in feat_ids])
    ax.set_xlabel('Activation')
    ax.set_title('Top SAE Features at Last Position')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

## Summary

| Function | Purpose |
|----------|--------|
| `polysemanticity_score()` | Measure polysemanticity of a single dimension |
| `feature_context_clusters()` | Cluster activating contexts to detect poly |
| `activation_decomposition()` | Decompose activation into SAE features |
| `feature_interference_matrix()` | Decoder direction overlap between features |
| `monosemanticity_ranking()` | Rank all dimensions by monosemanticity |